In [1]:
import pandas as pd

df = pd.read_csv('input_data/deliveries.csv')

df.sample(10)

,match_id,inning,batting_team,bowling_team,over,ball,batter,bowler,non_striker,batsman_runs,extra_runs,total_runs,extras_type,is_wicket,player_dismissed,dismissal_kind,fielder
41671,501198,1,Chennai Super Kings,Kolkata Knight Riders,12,6,MS Dhoni,Iqbal Abdulla,S Anirudha,1,0,1,NaN,0,NaN,NaN,NaN
114227,829749,1,Rajasthan Royals,Royal Challengers Bangalore,3,5,AM Rahane,Iqbal Abdulla,SR Watson,4,0,4,NaN,0,NaN,NaN,NaN
28982,419113,2,Kolkata Knight Riders,Chennai Super Kings,5,5,SC Ganguly,MS Gony,OA Shah,1,0,1,NaN,0,NaN,NaN,NaN
95881,729289,2,Delhi Daredevils,Kolkata Knight Riders,1,2,KD Karthik,M Morkel,MA Agarwal,0,0,0,NaN,0,NaN,NaN,NaN
79174,598009,2,Pune Warriors,Rajasthan Royals,6,5,LRPL Taylor,JP Faulkner,AJ Finch,1,0,1,NaN,0,NaN,NaN,NaN
112055,829731,1,Delhi Daredevils,Sunrisers Hyderabad,2,5,JP Duminy,B Kumar,SS Iyer,2,0,2,NaN,0,NaN,NaN,NaN
175053,1178418,2,Rajasthan Royals,Kolkata Knight Riders,0,4,AM Rahane,CR Brathwaite,SV Samson,4,0,4,NaN,0,NaN,NaN,NaN
97863,729307,1,Kings XI Punjab,Kolkata Knight Riders,2,6,V Sehwag,UT Yadav,WP Saha,0,0,0,NaN,0,NaN,NaN,NaN
239060,1359529,1,Chennai Super Kings,Delhi Capitals,9,3,AM Rahane,Kuldeep Yadav,MM Ali,1,0,1,NaN,0,NaN,NaN,NaN
215527,1304077,2,Lucknow Super Giants,Royal Challengers Bangalore,5,2,KH Pandya,Mohammed Siraj,KL Rahul,1,0,1,NaN,0,NaN,NaN,NaN


In [28]:
# PREPROCESSING DATA

match_ids = df.match_id.unique()

matches = {elem : pd.DataFrame() for elem in match_ids}

for key in matches.keys():
    matches[key] = df[:][df.match_id == key]



In [20]:
# DEFINE FUNCTION TO GET BATSMAN INNINGS

def get_batsman(match):
    '''Given a match, returns a dictionary of batsman with their 
    runs, balls faced, innings runs, batting position, and runs 
    when they came in and overs when they came in'''
    innings = match[match.inning == 1], match[match.inning == 2]
    players = {batter: {'name': batter,'runs': 0, 'balls': 0, 'innings_runs':0, 'batting_position': None, 'runs_come_in':None, 'overs_come_in': None, 'match_id':None} for batter in list(match.batter.unique())+list(match.non_striker.unique())}
    for inn in innings:
        total_runs = 0
        batsmanIn = []
        for _, ball in inn.iterrows():
            # Wide and no balls don't go to the batman's balls faced
            for bat in [ball.batter, ball.non_striker]:
                if bat not in batsmanIn:
                    batsmanIn.append(bat)
                    players[bat]['batting_position'] = len(batsmanIn)
                    players[bat]['runs_come_in'] = total_runs
                    players[bat]['overs_come_in'] = ball.over + ball.ball/6
                    players[bat]['match_id'] = ball.match_id
            if ball.extras_type == 'wides' or ball.extras_type == 'noballs':
                continue
            players[ball.batter]['runs'] += ball.batsman_runs
            players[ball.batter]['balls'] += 1

            total_runs += ball.batsman_runs
        for player in inn.batter.unique():
            players[player]['innings_runs'] = total_runs
        
    return players

# EXAMPLE
get_batsman(matches[335982])

{'SC Ganguly': {'name': 'SC Ganguly',
  'runs': 10,
  'balls': 12,
  'innings_runs': 205,
  'batting_position': 1,
  'runs_come_in': 0,
  'overs_come_in': 0.16666666666666666,
  'match_id': 335982},
 'BB McCullum': {'name': 'BB McCullum',
  'runs': 158,
  'balls': 73,
  'innings_runs': 205,
  'batting_position': 2,
  'runs_come_in': 0,
  'overs_come_in': 0.16666666666666666,
  'match_id': 335982},
 'RT Ponting': {'name': 'RT Ponting',
  'runs': 20,
  'balls': 20,
  'innings_runs': 205,
  'batting_position': 3,
  'runs_come_in': 51,
  'overs_come_in': 5.5,
  'match_id': 335982},
 'DJ Hussey': {'name': 'DJ Hussey',
  'runs': 12,
  'balls': 12,
  'innings_runs': 205,
  'batting_position': 4,
  'runs_come_in': 98,
  'overs_come_in': 12.333333333333334,
  'match_id': 335982},
 'Mohammad Hafeez': {'name': 'Mohammad Hafeez',
  'runs': 5,
  'balls': 3,
  'innings_runs': 205,
  'batting_position': 5,
  'runs_come_in': 155,
  'overs_come_in': 17.333333333333332,
  'match_id': 335982},
 'R Dravid

In [32]:
# GET ALL PLAYERS INNINGS FOR ALL MATCHES

all_players = {batter: [] for batter in list(df.batter.unique())+list(df.non_striker.unique())}

for match in matches.values():
    scores = get_batsman(match)
    for player, inn in scores.items():
        all_players[player].append(inn)

In [33]:
# COMBINE AND EXPORT TO CSV

every_bat = [a for b in all_players.values() for a in b]

every_player = [a['name'] for a in every_bat]
every_bat_runs = [a['runs'] for a in every_bat]
every_bat_balls = [a['balls'] for a in every_bat]
every_bat_innings_runs = [a['innings_runs'] for a in every_bat]
every_batting_position = [a['batting_position'] for a in every_bat]
every_runs_come_in = [a['runs_come_in'] for a in every_bat]
every_overs_come_in = [a['overs_come_in'] for a in every_bat]
every_match_id = [a['match_id'] for a in every_bat]

pd.DataFrame([every_player, every_bat_runs, every_bat_balls, every_bat_innings_runs, every_batting_position, every_runs_come_in, every_overs_come_in, every_match_id], index=['name', 'runs', 'balls', 'innings_runs', 'batting_position', 'runs_come_in', 'overs_come_in', 'match_id']).T.to_csv('batsman_runs.csv', index=False)